In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU count: 1
GPU name: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
dataset_clear_yaml_path = 'dataset_clear.yaml'

yaml_content = """
path: D:\Python\Train_model\YOLOv8s_DA\clear\clear
train: train/images
val: val/images
test: test/images
nc: 2
names: ['person', 'car']
"""

# Ghi nội dung này vào file
with open(dataset_clear_yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"Đã tạo file cấu hình dataset tại: {dataset_clear_yaml_path}")

Đã tạo file cấu hình dataset tại: dataset_clear.yaml


In [4]:
dataset_foggy_yaml_path = 'dataset_foggy.yaml'

yaml_content = """
path: D:\Python\Train_model\YOLOv8s_DA\foggy_anything\foggy_anything
train: train/images
val: val/images
test: test/images
nc: 2
names: ['person', 'car']
"""

# Ghi nội dung này vào file
with open(dataset_foggy_yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"Đã tạo file cấu hình dataset tại: {dataset_foggy_yaml_path}")

Đã tạo file cấu hình dataset tại: dataset_foggy.yaml


# Phần 1: Thiết lập môi trường và các thư viện cần thiết

In [5]:
import os
import random
import time
import warnings
from functools import partial

import numpy as np
import torch
import torch.nn as nn
from torch import distributed as dist
from torch.nn import functional as F
from torch.nn.utils import spectral_norm
from torchvision.ops import box_convert, roi_align
from ultralytics.data.utils import check_det_dataset
from ultralytics.models import YOLO
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.utils import LOGGER, RANK, TQDM, colorstr, clean_url, emojis
import ultralytics.utils.callbacks.tensorboard as tb_module


def seed_everything(seed=9527):
    """
    Cố định tất cả các seed ngẫu nhiên để đảm bảo tính tái lập (reproducibility) của quá trình huấn luyện.
    Điều này rất quan trọng để có thể so sánh kết quả giữa các lần chạy khác nhau.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Phần 2: Các thành phần Domain Adaptation (DANN)

In [6]:
class GradientReversalFunction(torch.autograd.Function):
    """
    Hàm đảo ngược gradient.
    Trong quá trình lan truyền tiến (forward), nó hoạt động như một hàm nhận dạng.
    Trong quá trình lan truyền ngược (backward), nó nhân gradient với -alpha,
    đảo ngược hướng của gradient.
    """
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None


class GradientReversalLayer(torch.nn.Module):
    """
    Lớp PyTorch wrapper cho GradientReversalFunction.
    """
    def __init__(self, alpha=1.):
        super(GradientReversalLayer, self).__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)


class SpectralLinear(nn.Linear):
    """Lớp tuyến tính (Linear) với chuẩn hóa phổ (Spectral Normalization)"""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, name='weight', n_power_iterations=1, dim=0, eps=1e-12)


class SpectralConv2d(nn.Conv2d):
    """Lớp tích chập 2D (Conv2d) với chuẩn hóa phổ (Spectral Normalization)"""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, name='weight', n_power_iterations=1, dim=0, eps=1e-12)


class DiscriminatorHead(nn.Module):
    """
    Phần đầu của bộ phân loại. Nó nhận các đặc trưng và dự đoán miền của chúng.
    Được thiết kế với nhiều lớp tuyến tính, sử dụng chuẩn hóa phổ để cải thiện
    sự ổn định khi huấn luyện.
    """
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h // 2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim_h // 2, dim_h // 2),
                nn.LeakyReLU(0.2, inplace=True),
            ) for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h // 2 * 4, dim_h // 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(dim_h // 2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        x = x.split(x.shape[1] // 2, dim=1)
        xs = [x[0]]
        for m in self.neck:
            x = m(x[1]) if isinstance(x, tuple) else m(x)
            xs.append(x)
        x = torch.cat(xs, dim=1)
        return self.head(x)


class Discriminator(nn.Module):
    """
    Bộ phân loại (Discriminator) chính.
    Bao gồm lớp đảo ngược gradient và các lớp tích chập để xử lý các đặc trưng
    từ mô hình YOLOv8 và đưa chúng vào DiscriminatorHead.
    """
    def __init__(self, chs=None, amp=False):
        super().__init__()
        if chs is None:
            chs = [64, 128, 256] # Kích thước kênh mặc định của các lớp được hook trên YOLOv8s
        self.chs = chs
        self.f_len = len(chs)
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        # Các lớp để trích xuất đặc trưng
        self.p = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i == 0 else chs[i] * 2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64),
                nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(32),
                nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i + 1] if i + 1 < len(chs) else chs[i], kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(chs[i + 1] if i + 1 < len(chs) else chs[i]),
                nn.SiLU(inplace=True),
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, fs: list[torch.tensor]):
        with torch.cuda.amp.autocast(self.amp):
            assert len(fs) == self.f_len, f'Expected {self.f_len} feature maps, got {len(fs)}'
            fs = [self.grl(f) for f in fs] # Áp dụng lớp đảo ngược gradient
            x = self.p[0](fs[0])
            for i in range(1, len(fs)):
                x = torch.cat((x, fs[i]), dim=1)
                x = self.p[i](x)
            return self.head(x)

# Phần 3: Lớp Custom Trainer

In [7]:
from torch.utils.data import DataLoader

class CustomTrainer(DetectionTrainer):
    """
    Huấn luyện viên tùy chỉnh để thực hiện Domain Adaptation.
    Kế thừa từ `DetectionTrainer` của `ultralytics` và thêm logic cho DANN.
    """
    def __init__(self, target_domain_data_cfg, *args, **kwargs):
        super(CustomTrainer, self).__init__(*args, **kwargs)
        self.t_data_cfg = target_domain_data_cfg
        
        self.t_iter = None
        self.t_train_loader = None
        self.model_hook_handler = []
        self.model_hook_layer_idx: list[int] = [2, 4, 6]
        self.roi_size = list(reversed([20 * 2 ** i for i in range(len(self.model_hook_layer_idx))]))
        self.model_hooked_features: None | list[torch.tensor] = None
        self.discriminator_model = None
        self.projection_model = None
        self.additional_models = []
        
        # Thêm callback để khởi tạo mô hình phụ trợ
        self.add_callback('on_train_start', self.init_helper_model)


    def init_helper_model(self, *args, **kwargs):
        """Khởi tạo mô hình phân loại miền (discriminator) và thêm vào danh sách các mô hình phụ trợ."""
        if not hasattr(self, 'writer') or self.writer is None:
            try:
                from torch.utils.tensorboard import SummaryWriter
                self.writer = SummaryWriter(self.save_dir) if self.args.save else None
            except ImportError:
                self.writer = None
                LOGGER.warning('TensorBoard không được cài đặt, không thể ghi log các chỉ số huấn luyện. Cài đặt bằng `pip install tensorboard`.')
        
        self.discriminator_model = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator_model)

    def get_t_batch(self):
        """Lấy một batch dữ liệu từ miền đích."""
        if self.t_iter is None:
            # Sửa lỗi: Bây giờ mới kiểm tra và tải dữ liệu miền đích khi thực sự cần.
            # Điều này sẽ giúp tránh lỗi nếu đường dẫn val không tồn tại.
            try:
                self.t_data = check_det_dataset(self.t_data_cfg)
            except Exception as e:
                raise RuntimeError(emojis(f"Dataset '{clean_url(self.t_data_cfg)}' error ❌ {e}")) from e
            # Sửa lỗi: Truyền đúng đường dẫn tới thư mục images thay vì toàn bộ dictionary.
            t_train_img_path = str(self.t_data['path'] / self.t_data['train'])
            # Thêm `batch=self.batch_size` để khắc phục lỗi `TypeError`
            t_trainset = self.build_dataset(t_train_img_path, 'train', batch=self.batch_size)
            # Khắc phục lỗi AttributeError bằng cách tạo DataLoader thủ công
            self.t_train_loader = DataLoader(
                t_trainset,
                batch_size=self.batch_size,
                num_workers=self.args.workers,
                collate_fn=getattr(t_trainset, 'collate_fn', None), # Lấy collate_fn từ dataset
            )
            self.t_iter = iter(self.t_train_loader)
        try:
            batch = next(self.t_iter)
        except StopIteration:
            self.t_iter = iter(self.t_train_loader)
            batch = next(self.t_iter)
        return batch

    def activate_hook(self, layer_indices: list[int] = None):
        """
        Kích hoạt các hook để lấy các đặc trưng từ các lớp cụ thể của mô hình.
        Đây là cách chúng ta lấy các đặc trưng trung gian để đưa vào bộ phân loại.
        """
        if layer_indices is not None:
            self.model_hook_layer_idx = layer_indices
        self.model_hooked_features = [None for _ in self.model_hook_layer_idx]
        self.model_hook_handler = \
            [self.model.model[l].register_forward_hook(self.hook_fn(i)) for i, l in enumerate(self.model_hook_layer_idx)]

    def deactivate_hook(self):
        """Tắt và gỡ bỏ tất cả các hook."""
        if self.model_hook_handler is not None:
            for hook in self.model_hook_handler:
                hook.remove()
            self.model_hooked_features = None
            self.model_hook_handler = []

    def hook_fn(self, hook_idx):
        """Hàm hook để lưu trữ các đặc trưng."""
        def hook(m, i, o):
            self.model_hooked_features[hook_idx] = o
        return hook

    def get_dis_output_from_hooked_features(self, batch):
        """Trích xuất các đặc trưng ROI từ các đặc trưng đã được hook và đưa vào bộ phân loại."""
        if self.model_hooked_features is not None:
            bbox_batch_idx = batch['batch_idx'].unsqueeze(-1)
            bbox = batch['bboxes']
            bbox = box_convert(bbox, 'cxcywh', 'xyxy')
            rois = []
            for fidx, f in enumerate(self.model_hooked_features):
                f_bbox = bbox * f.shape[-1]
                f_bbox = torch.cat((bbox_batch_idx, f_bbox), dim=-1)
                f_roi = roi_align(f, f_bbox.to(f.device), output_size=self.roi_size[fidx], aligned=True)
                rois.append(f_roi)
            dis_output = self.discriminator_model(rois)
            return dis_output
        else:
            return None

    def optimizer_step(self, optims: None | list[torch.optim.Optimizer] = None):
        """Thực hiện một bước tối ưu hóa cho cả mô hình chính và mô hình phụ trợ."""
        self.scaler.unscale_(self.optimizer)
        if optims is not None:
            for o in optims:
                if o.param_groups[0]['params'][0].grad is not None:
                    self.scaler.unscale_(o)
        max_norm = 10.0
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=max_norm)
        if len(self.additional_models) > 0:
            for m in self.additional_models:
                torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=max_norm * 2)
        self.scaler.step(self.optimizer)
        if optims is not None:
            for o in optims:
                if o.param_groups[0]['params'][0].grad is not None:
                    self.scaler.step(o)
        self.scaler.update()
        self.optimizer.zero_grad()
        if optims is not None:
            for o in optims:
                o.zero_grad()
        if self.ema:
            self.ema.update(self.model)

    def _do_train(self, world_size=1):
        """
        Thực hiện vòng lặp huấn luyện chính.
        Đây là nơi logic cho domain adaptation được tích hợp vào quá trình huấn luyện của YOLO.
        """
        if world_size > 1:
            self._setup_ddp(world_size)
        self._setup_train()
        self.epoch_time = None
        self.epoch_time_start = time.time()
        self.train_time_start = time.time()
        nb = len(self.train_loader)
        nw = max(round(self.args.warmup_epochs * nb), 100) if self.args.warmup_epochs > 0 else -1
        last_opt_step = -1
        self.run_callbacks('on_train_start')
        LOGGER.info(f'Image sizes {self.args.imgsz} train, {self.args.imgsz} val\n'
                    f'Using {self.train_loader.num_workers * (world_size or 1)} dataloader workers\n'
                    f"Logging results to {colorstr('bold', self.save_dir)}\n"
                    f'Starting training for {self.epochs} epochs...')
        
        LOGGER.info('Bắt đầu tải dữ liệu và chuẩn bị dataloader. Quá trình này có thể mất một lúc...')
        
        if self.args.close_mosaic:
            base_idx = (self.epochs - self.args.close_mosaic) * nb
            self.plot_idx.extend([base_idx, base_idx + 1, base_idx + 2])
        epoch = self.epochs
        self.activate_hook()
        for epoch in range(self.start_epoch, self.epochs):
            self.epoch = epoch
            self.run_callbacks('on_train_epoch_start')
            self.model.train()
            if RANK != -1:
                self.train_loader.sampler.set_epoch(epoch)
            pbar = enumerate(self.train_loader)
            if epoch == (self.epochs - self.args.close_mosaic):
                LOGGER.info('Closing dataloader mosaic')
                if hasattr(self.train_loader.dataset, 'mosaic'):
                    self.train_loader.dataset.mosaic = False
                if hasattr(self.train_loader.dataset, 'close_mosaic'):
                    self.train_loader.dataset.close_mosaic(hyp=self.args)
                self.train_loader.reset()
            if RANK in (-1, 0):
                LOGGER.info(self.progress_string())
                pbar = TQDM(enumerate(self.train_loader), total=nb)
            self.tloss = None
            self.optimizer.zero_grad()
            for i, batch in pbar:
                self.run_callbacks('on_train_batch_start')
                ni = i + nb * epoch
                if ni <= nw:
                    xi = [0, nw]
                    self.accumulate = max(1, np.interp(ni, xi, [1, self.args.nbs / self.batch_size]).round())
                    for j, x in enumerate(self.optimizer.param_groups):
                        x['lr'] = np.interp(
                            ni, xi, [self.args.warmup_bias_lr if j == 0 else 0.0, x['initial_lr'] * self.lf(epoch)])
                        if 'momentum' in x:
                            x['momentum'] = np.interp(ni, xi, [self.args.warmup_momentum, self.args.momentum])
                with torch.cuda.amp.autocast(self.amp):
                    batch = self.preprocess_batch(batch)
                    self.loss, self.loss_items = self.model(batch)
                    if RANK != -1:
                        self.loss *= world_size
                    self.tloss = (self.tloss * i + self.loss_items) / (i + 1) if self.tloss is not None else self.loss_items
                    source_critics = self.get_dis_output_from_hooked_features(batch)
                    t_batch = self.get_t_batch()
                    t_batch = self.preprocess_batch(t_batch)
                    t_loss, t_loss_item = self.model(t_batch)
                    target_critics = self.get_dis_output_from_hooked_features(t_batch)
                    self.loss += t_loss
                    if 6 < epoch < self.args.epochs - 50:
                        threshold = 20
                        loss_d = (F.relu(torch.ones_like(source_critics) * threshold + source_critics)).mean()
                        loss_d += (F.relu(torch.ones_like(target_critics) * threshold - target_critics)).mean()
                    else:
                        loss_d = 0
                    self.loss += loss_d * 2
                self.scaler.scale(self.loss.sum()).backward()
                if ni - last_opt_step >= self.accumulate:
                    self.optimizer_step(optims=[self.discriminator_model.optim])
                    last_opt_step = ni
                mem = f'{torch.cuda.memory_reserved() / 1E9 if torch.cuda.is_available() else 0:.3g}G'
                loss_len = self.tloss.shape[0] if len(self.tloss.size()) else 1
                losses = self.tloss if loss_len > 1 else torch.unsqueeze(self.tloss, 0)
                if RANK in (-1, 0):
                    pbar.set_description(
                        ('%11s' * 2 + '%11.4g' * (2 + loss_len)) %
                        (f'{epoch + 1}/{self.epochs}', mem, *losses, batch['cls'].shape[0], batch['img'].shape[-1]))
                    self.run_callbacks('on_batch_end')
                    if self.writer:
                        self.writer.add_scalar('train/critic-source', source_critics.mean(), ni)
                        self.writer.add_scalar('train/critic-target', target_critics.mean(), ni)
                    if self.args.plots and ni in self.plot_idx:
                        self.plot_training_samples(batch, ni)
                self.run_callbacks('on_train_batch_end')
            self.lr = {f'lr/pg{ir}': x['lr'] for ir, x in enumerate(self.optimizer.param_groups)}
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                self.scheduler.step()
            self.run_callbacks('on_train_epoch_end')
            if RANK in (-1, 0):
                self.ema.update_attr(self.model, include=['yaml', 'nc', 'args', 'names', 'stride', 'class_weights'])
                final_epoch = (epoch + 1 == self.epochs) or self.stopper.possible_stop
                if self.args.val or final_epoch:
                    self.metrics, self.fitness = self.validate()
                self.save_metrics(metrics={**self.label_loss_items(self.tloss), **self.metrics, **self.lr})
                self.stop = self.stopper(epoch + 1, self.fitness)
                if self.args.save or (epoch + 1 == self.epochs):
                    self.deactivate_hook()
                    self.save_model()
                    self.run_callbacks('on_model_save')
                    self.activate_hook()
            tnow = time.time()
            self.epoch_time = tnow - self.epoch_time_start
            self.epoch_time_start = tnow
            self.run_callbacks('on_fit_epoch_end')
            torch.cuda.empty_cache()
            if RANK != -1:
                broadcast_list = [self.stop if RANK == 0 else None]
                dist.broadcast_object_list(broadcast_list, 0)
                if RANK != 0:
                    self.stop = broadcast_list[0]
            if self.stop:
                break
        if RANK in (-1, 0):
            LOGGER.info(f'\n{epoch - self.start_epoch + 1} epochs completed in '
                        f'{(time.time() - self.train_time_start) / 3600:.3f} hours.')
            self.final_eval()
            if self.args.plots:
                self.plot_metrics()
            self.run_callbacks('on_train_end')
        self.deactivate_hook()
        torch.cuda.empty_cache()
        self.run_callbacks('teardown')

# Phần 4: Vòng lặp huấn luyện chính

In [12]:
model = YOLO('D:\Python\Train_model\YOLOv8s_DA\YOLOv8sDA_continue.pt')

# Đánh giá mô hình đã được huấn luyện với DA trên miền trong
print("Đánh giá model với clear images: ")
model.val(data='D:\Python\Train_model\YOLOv8s_DA\dataset_clear.yaml', name='val_RMD_clear', split='test')
    
# Đánh giá mô hình đã được huấn luyện với DA trên miền sương mù
print("Đánh giá model với foggy images: ")
model.val(data='D:\Python\Train_model\YOLOv8s_DA\dataset_foggy.yaml', name='val_RMD_foggy', split='test')

Đánh giá model với clear images: 
Ultralytics 8.3.202  Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 72 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 2.44.8 ms, read: 492.745.5 MB/s, size: 2254.4 KB)
val: Scanning D:\Python\Train_model\YOLOv8s_DA\clear\clear\test\labels... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 273.2it/s 0.4s1s
val: New cache created: D:\Python\Train_model\YOLOv8s_DA\clear\clear\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.5it/s 2.8s0.3ss
                   all        100       1673      0.746      0.515      0.579      0.367
                person         85        710      0.693      0.394      0.455       0.24
                   car         97        963      0.799      0.636      0.702      0.493
Speed: 0.8ms preprocess, 9.2ms inference, 0.0ms loss, 6.5

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000239F124E410>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

In [ ]:
def main():
    """
    Hàm chính để bắt đầu quá trình huấn luyện và đánh giá.
    Đầu tiên, nó huấn luyện mô hình với Domain Adaptation, sau đó
    huấn luyện và đánh giá các mô hình baseline để so sánh.
    """
    # Thiết lập Hyperparameters và seed
    seed = 9527
    kwargs = {
        'imgsz': 640,
        'epochs': 80,
        'val': True,
        'workers': 0,
        'batch': 8,
        'seed': seed,
    }
    seed_everything(seed)
    
    # === Huấn luyện với Domain Adaptation (DANN) ===
    # Tải mô hình YOLOv8s pretrained
    model = YOLO("D:\Python\Train_model\YOLOv8s_DA\yolov8s_clear_20ep.pt")
    # model = YOLO('yolov8s.yaml').load('weights/yolov8s.pt')
    # Khởi tạo trainer tùy chỉnh với miền đích là dữ liệu sương mù (fog)
    custom_trainer = partial(CustomTrainer, target_domain_data_cfg='D:\Python\Train_model\YOLOv8s_DA\dataset_foggy.yaml')
    # Bắt đầu huấn luyện, sử dụng dữ liệu nguồn là dữ liệu trời trong (clear)
    model.train(custom_trainer, data='D:\Python\Train_model\YOLOv8s_DA\dataset_clear.yaml', name='train_RMD', patience=10, **kwargs)
    
    # Đánh giá mô hình đã được huấn luyện với DA trên miền trong
    print("Đánh giá model với clear images: ")
    model.val(data='D:\Python\Train_model\YOLOv8s_DA\dataset_clear.yaml', name='val_RMD_clear')
    
    # Đánh giá mô hình đã được huấn luyện với DA trên miền sương mù
    print("Đánh giá model với foggy images: ")
    model.val(data='D:\Python\Train_model\YOLOv8s_DA\dataset_foggy.yaml', name='val_RMD_foggy')


if __name__ == '__main__':
    main()


New https://pypi.org/project/ultralytics/8.3.203 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.202  Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Python\Train_model\YOLOv8s_DA\dataset_clear.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\Python\Train_model\YOLOv8s_DA\yolov8s_clear_20ep.pt,

KeyboardInterrupt: 